# 03.0 Autoencoder baseline

## Notebook overview
This notebook trains and evaluates the reconstruction-based autoencoder baseline using the fixed split manifest from notebook 01, then merges the result with the ImageNet baseline tables from notebook 02.

## Purpose
- train one small convolutional autoencoder per category using only normal training images
- evaluate the autoencoder on the shared test split with the same image-level and pixel-level metrics used in the baseline comparison
- save autoencoder-only category and mean tables, checkpoints, figures, and run metadata
- create a merged three-method baseline table so later notebooks can compare ImageNet PatchCore, ImageNet PaDiM, and the autoencoder together

## Process
1. load the saved split manifest and leakage report from notebook 01
2. load the ImageNet baseline tables from notebook 02
3. build the shared dataset and metric helpers for reconstruction-based evaluation
4. train and evaluate one autoencoder per category
5. save autoencoder artefacts and a merged baseline comparison table

## Notes
- run `01_dataset_audit_and_splits.ipynb` first
- run `02_baseline_imagenet_models.ipynb` first so the merged baseline table includes all three baseline methods
- this notebook keeps only the code needed for the reconstruction baseline and removes unrelated ImageNet and SimCLR logic


In [ ]:
#------------------------------------------------------------------------------
# 1.0 Imports and notebook helpers
#------------------------------------------------------------------------------
import os
import sys
import json
import time
import math
import random
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms

from sklearn.metrics import roc_auc_score, average_precision_score

from IPython.display import display


# Print a clean section banner so notebook output is easy to scan.
def print_banner(text):
    print("\n" + "=" * 90)
    print(text)
    print("=" * 90)


# Create a parent folder before saving an artefact.
def ensure_parent(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


# Save JSON with overwrite behaviour.
def save_json_overwrite(obj, path):
    path = ensure_parent(path)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


# Save a DataFrame to CSV with overwrite behaviour.
def save_csv_overwrite(df, path):
    path = ensure_parent(path)
    df.to_csv(path, index=False)


# Read JSON from disk.
def read_json(path):
    with open(Path(path), "r") as f:
        return json.load(f)


# Return a file size in MB.
def file_size_mb(path):
    path = Path(path)
    return path.stat().st_size / (1024 ** 2) if path.exists() else np.nan


# Set Python, NumPy, and PyTorch seeds.
def set_seed(seed, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


# Reset peak GPU memory before timing a stage.
def reset_peak_gpu_memory():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


# Return peak GPU memory in MB if CUDA is available.
def get_peak_gpu_memory_mb():
    if not torch.cuda.is_available():
        return np.nan
    return torch.cuda.max_memory_allocated() / (1024 ** 2)


# 2.0 Run settings

## Purpose
- define the reconstruction baseline settings in one place
- make the notebook portable across Kaggle and local environments
- keep paths to prior notebook artefacts explicit so dependency errors are obvious
- save outputs in a clean structure that can be committed to GitHub


In [ ]:
#------------------------------------------------------------------------------
# 2.1 Core configuration and output paths
#------------------------------------------------------------------------------
NOTEBOOK_ID = "03_autoencoder_baseline"
RUN_MODE = "full"                  # "debug" or "full"
SEED = 42
DETERMINISTIC_DEBUG = False
set_seed(SEED, deterministic=(RUN_MODE == "debug" and DETERMINISTIC_DEBUG))

DEBUG_CATEGORIES = ["bottle", "carpet", "tile", "transistor"]
CATEGORIES_ALL = [
    "bottle",
    "cable",
    "capsule",
    "carpet",
    "grid",
    "hazelnut",
    "leather",
    "metal_nut",
    "pill",
    "screw",
    "tile",
    "toothbrush",
    "transistor",
    "wood",
    "zipper",
]
CATEGORIES_ACTIVE = DEBUG_CATEGORIES if RUN_MODE == "debug" else CATEGORIES_ALL
QUAL_CATEGORY = next((cat for cat in ["carpet", "tile", "bottle", "transistor"] if cat in CATEGORIES_ACTIVE), CATEGORIES_ACTIVE[0])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_COUNT = torch.cuda.device_count()

IMG_SIZE = 224
BATCH_SIZE_TRAIN = 32
BATCH_SIZE_TEST = 16
AE_EPOCHS = 3 if RUN_MODE == "debug" else 5
AE_LR = 1e-3
PATCH_SCORE_TOPK = 200

CPU_COUNT = os.cpu_count() or 2
NUM_WORKERS_BASE = min(4, max(2, CPU_COUNT // 2))
NUM_WORKERS = NUM_WORKERS_BASE if DEVICE == "cuda" else 0
PERSISTENT_WORKERS = NUM_WORKERS > 0
PREFETCH_FACTOR = 2 if NUM_WORKERS > 0 else None

if Path("/kaggle/working").exists():
    WORK_ROOT = Path("/kaggle/working/industrial_anomaly_detection_mvtec")
else:
    WORK_ROOT = Path.cwd() / "industrial_anomaly_detection_mvtec"

NOTEBOOK_ROOT = WORK_ROOT / NOTEBOOK_ID / RUN_MODE
FIGURES_DIR = NOTEBOOK_ROOT / "figures"
TABLES_DIR = NOTEBOOK_ROOT / "tables"
JSON_DIR = NOTEBOOK_ROOT / "json"
CHECKPOINTS_DIR = NOTEBOOK_ROOT / "checkpoints"

for folder in [FIGURES_DIR, TABLES_DIR, JSON_DIR, CHECKPOINTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

NOTEBOOK_01_ROOT = WORK_ROOT / "01_dataset_audit_and_splits" / RUN_MODE
NOTEBOOK_02_ROOT = WORK_ROOT / "02_baseline_imagenet_models" / RUN_MODE

SPLIT_MANIFEST_PATH = NOTEBOOK_01_ROOT / "json" / "split_manifest.json"
LEAKAGE_REPORT_PATH = NOTEBOOK_01_ROOT / "json" / "leakage_report.json"

TABLE_BASELINES_IMAGENET_CATEGORY_PATH = NOTEBOOK_02_ROOT / "tables" / "table_baselines_imagenet_category.csv"
TABLE_BASELINES_IMAGENET_MEAN_PATH = NOTEBOOK_02_ROOT / "tables" / "table_baselines_imagenet_mean.csv"
TABLE_BASELINES_IMAGENET_FULL_PATH = NOTEBOOK_02_ROOT / "tables" / "table_baselines_imagenet_full.csv"

RUN_METADATA_PATH = JSON_DIR / "run_metadata.json"
AUTOENCODER_SETTINGS_PATH = JSON_DIR / "autoencoder_settings.json"
TABLE_BASELINES_AUTOENCODER_CATEGORY_PATH = TABLES_DIR / "table_baselines_autoencoder_category.csv"
TABLE_BASELINES_AUTOENCODER_MEAN_PATH = TABLES_DIR / "table_baselines_autoencoder_mean.csv"
TABLE_BASELINES_AUTOENCODER_FULL_PATH = TABLES_DIR / "table_baselines_autoencoder_full.csv"
TABLE_BASELINES_ALL_CATEGORY_PATH = TABLES_DIR / "table_baselines_all_category.csv"
TABLE_BASELINES_ALL_MEAN_PATH = TABLES_DIR / "table_baselines_all_mean.csv"
TABLE_BASELINES_ALL_FULL_PATH = TABLES_DIR / "table_baselines_all_full.csv"
FIG_BASELINES_AUTOENCODER_PATH = FIGURES_DIR / "fig_baselines_autoencoder.png"
FIG_BASELINES_ALL_PATH = FIGURES_DIR / "fig_baselines_all.png"
FIG_HEATMAPS_AUTOENCODER_PATH = FIGURES_DIR / "fig_heatmaps_autoencoder.png"
FIG_AE_TRAINING_LOSS_PATH = FIGURES_DIR / "fig_autoencoder_training_loss.png"

RUN_METADATA = {
    "notebook_id": NOTEBOOK_ID,
    "run_mode": RUN_MODE,
    "seed": SEED,
    "device": DEVICE,
    "gpu_count": GPU_COUNT,
    "image_size": IMG_SIZE,
    "ae_epochs": AE_EPOCHS,
    "ae_lr": AE_LR,
    "categories_active": CATEGORIES_ACTIVE,
    "work_root": str(WORK_ROOT),
}
save_json_overwrite(RUN_METADATA, RUN_METADATA_PATH)

print_banner("Run configuration")
print("RUN_MODE         :", RUN_MODE)
print("SEED             :", SEED)
print("DEVICE           :", DEVICE)
print("GPU_COUNT        :", GPU_COUNT)
print("AE_EPOCHS        :", AE_EPOCHS)
print("AE_LR            :", AE_LR)
print("ACTIVE CATEGORIES:", CATEGORIES_ACTIVE)
print("WORK_ROOT        :", WORK_ROOT)


# 3.0 Dataset and prior artefacts

## Purpose
- resolve the dataset location
- load the saved split manifest from notebook 01
- verify that notebook 01 reported zero exact leakage
- load the ImageNet baseline tables from notebook 02 so the merged baseline comparison can be built safely


In [ ]:
#------------------------------------------------------------------------------
# 3.1 Resolve the dataset path and load prior notebook artefacts
#------------------------------------------------------------------------------
DATASET_CANDIDATES = [
    Path(os.environ.get("MVTEC_DIR", "")) if os.environ.get("MVTEC_DIR") else None,
    Path("/kaggle/input/datasets/ipythonx/mvtec-ad"),
    Path("/kaggle/input/mvtec-ad"),
    Path.cwd() / "data" / "raw" / "mvtec-ad",
]

MVTEC_DIR = None
for candidate in DATASET_CANDIDATES:
    if candidate is not None and candidate.exists():
        MVTEC_DIR = candidate
        break

if MVTEC_DIR is None:
    raise FileNotFoundError(
        "MVTec AD dataset path was not found. Set MVTEC_DIR as an environment variable "
        "or attach the dataset in Kaggle before running the notebook."
    )

if not SPLIT_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Split manifest was not found at {SPLIT_MANIFEST_PATH}. "
        "Run 01_dataset_audit_and_splits.ipynb first."
    )

if not LEAKAGE_REPORT_PATH.exists():
    raise FileNotFoundError(
        f"Leakage report was not found at {LEAKAGE_REPORT_PATH}. "
        "Run 01_dataset_audit_and_splits.ipynb first."
    )

if not TABLE_BASELINES_IMAGENET_CATEGORY_PATH.exists():
    raise FileNotFoundError(
        f"ImageNet baseline category table was not found at {TABLE_BASELINES_IMAGENET_CATEGORY_PATH}. "
        "Run 02_baseline_imagenet_models.ipynb first."
    )

if not TABLE_BASELINES_IMAGENET_MEAN_PATH.exists():
    raise FileNotFoundError(
        f"ImageNet baseline mean table was not found at {TABLE_BASELINES_IMAGENET_MEAN_PATH}. "
        "Run 02_baseline_imagenet_models.ipynb first."
    )

splits = read_json(SPLIT_MANIFEST_PATH)
leakage_report = read_json(LEAKAGE_REPORT_PATH)
df_baselines_imagenet_category = pd.read_csv(TABLE_BASELINES_IMAGENET_CATEGORY_PATH)
df_baselines_imagenet_mean = pd.read_csv(TABLE_BASELINES_IMAGENET_MEAN_PATH)

if not leakage_report.get("all_checks_zero", False):
    raise AssertionError(
        "Notebook 01 reported non-zero leakage checks. Fix that before running the autoencoder notebook."
    )

missing_categories = sorted(set(CATEGORIES_ACTIVE) - set(splits.keys()))
if len(missing_categories) > 0:
    raise ValueError(
        f"The split manifest does not contain the required categories for this run: {missing_categories}"
    )

available_categories = sorted([path.name for path in MVTEC_DIR.iterdir() if path.is_dir()])
imagenet_methods = sorted(df_baselines_imagenet_category["method"].dropna().unique().tolist())

if sorted(set(CATEGORIES_ACTIVE) - set(df_baselines_imagenet_category["category"].unique().tolist())):
    raise ValueError("Notebook 02 category table does not contain all active categories for this run.")

print_banner("Loaded inputs")
print("MVTEC_DIR                     :", MVTEC_DIR)
print("Available categories          :", available_categories)
print("Split manifest path           :", SPLIT_MANIFEST_PATH)
print("Leakage report path           :", LEAKAGE_REPORT_PATH)
print("ImageNet category table path  :", TABLE_BASELINES_IMAGENET_CATEGORY_PATH)
print("ImageNet mean table path      :", TABLE_BASELINES_IMAGENET_MEAN_PATH)
print("ImageNet methods loaded       :", imagenet_methods)

display(df_baselines_imagenet_category.head())


# 4.0 Shared datasets, transforms, and evaluation helpers

## Purpose
- define the dataset wrapper and dataloader logic for reconstruction-based inputs
- keep all shared metric and plotting helpers in one place
- ensure the autoencoder is evaluated with the same core image-level and pixel-level metrics used elsewhere in the project


In [ ]:
#------------------------------------------------------------------------------
# 4.1 Shared transforms, dataset class, loaders, metrics, and plotting helpers
#------------------------------------------------------------------------------
tfm_ae = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])


# Load one RGB image from disk.
def load_rgb(path):
    return Image.open(path).convert("RGB")


# Load one binary mask from disk, or return an all-zero mask for normal images.
def load_mask(path):
    if path is None or (isinstance(path, float) and np.isnan(path)):
        return torch.zeros((1, IMG_SIZE, IMG_SIZE), dtype=torch.float32)

    mask = Image.open(path).convert("L").resize((IMG_SIZE, IMG_SIZE), resample=Image.NEAREST)
    mask = (np.array(mask, dtype=np.float32) > 0).astype(np.float32)
    return torch.from_numpy(mask)[None, :, :]


# Return items in a consistent format across train, validation, and test splits.
class MvtecDataset(Dataset):
    def __init__(self, items, mode, img_tfm):
        self.items = items
        self.mode = mode
        self.img_tfm = img_tfm

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]

        if self.mode in ["train_good", "val_good"]:
            img_path = item
            label = 0
            mask_path = None
        else:
            img_path = item["img_path"]
            label = int(item["label"])
            mask_path = item.get("mask_path", None)

        image = self.img_tfm(load_rgb(img_path))
        mask = load_mask(mask_path)
        return image, int(label), mask, str(img_path)


# Build a DataLoader using settings matched to the active runtime.
def make_loader(items, mode, img_tfm, batch_size, shuffle):
    dataset = MvtecDataset(items=items, mode=mode, img_tfm=img_tfm)

    loader_kwargs = dict(
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
    )

    if NUM_WORKERS > 0:
        loader_kwargs["persistent_workers"] = PERSISTENT_WORKERS
        loader_kwargs["prefetch_factor"] = PREFETCH_FACTOR

    return DataLoader(dataset, **loader_kwargs)


# Build the train, validation, and test loaders for one category.
def make_split_loaders(category):
    train_loader = make_loader(
        items=splits[category]["train_good"],
        mode="train_good",
        img_tfm=tfm_ae,
        batch_size=BATCH_SIZE_TRAIN,
        shuffle=True,
    )

    val_loader = make_loader(
        items=splits[category]["val_good"],
        mode="val_good",
        img_tfm=tfm_ae,
        batch_size=BATCH_SIZE_TEST,
        shuffle=False,
    )

    test_loader = make_loader(
        items=splits[category]["test"],
        mode="test",
        img_tfm=tfm_ae,
        batch_size=BATCH_SIZE_TEST,
        shuffle=False,
    )

    return train_loader, val_loader, test_loader


# Compute image ROC-AUC while handling single-class edge cases.
def safe_roc_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    return np.nan if len(np.unique(y_true)) < 2 else roc_auc_score(y_true, y_score)


# Compute image PR-AUC while handling single-class edge cases.
def safe_pr_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    return np.nan if len(np.unique(y_true)) < 2 else average_precision_score(y_true, y_score)


# Flatten masks and heatmaps so pixel ROC-AUC can be measured.
def pixel_roc_auc(masks_list, heatmaps_list):
    y_true = np.concatenate([mask.reshape(-1) for mask in masks_list]).astype(int)
    y_score = np.concatenate([heat.reshape(-1) for heat in heatmaps_list]).astype(float)
    return np.nan if len(np.unique(y_true)) < 2 else roc_auc_score(y_true, y_score)


# Rescale an array to the 0 to 1 range for plotting.
def norm_01(x):
    x = np.asarray(x, dtype=np.float32)
    return (x - x.min()) / (x.max() - x.min() + 1e-8)


# Convert a tensor image to display format.
def tensor_to_display(img_tensor):
    img = img_tensor.detach().cpu().permute(1, 2, 0).numpy()
    return np.clip(img, 0.0, 1.0)


# Draw an image with an optional heatmap overlay.
def overlay(ax, img_tensor, heat_2d=None, title=""):
    ax.imshow(tensor_to_display(img_tensor))
    if heat_2d is not None:
        ax.imshow(norm_01(heat_2d), alpha=0.45)
    ax.set_title(title, fontsize=10)
    ax.axis("off")


# Show a mask with a clean grayscale colourmap.
def show_mask(ax, mask_2d, title=""):
    ax.imshow(mask_2d, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title, fontsize=10)
    ax.axis("off")


# Pick a few normal and anomalous items for the qualitative figure.
def select_qual_items(test_items, n_good=3, n_bad=3):
    good_items = [item for item in test_items if int(item["label"]) == 0][:n_good]
    bad_items = [item for item in test_items if int(item["label"]) == 1][:n_bad]
    return good_items + bad_items


# 5.0 Autoencoder helpers

## Purpose
- define the small convolutional autoencoder used as the reconstruction baseline
- keep training and evaluation helpers short and focused
- make saved checkpoints and summary tables easy to interpret later


In [ ]:
#------------------------------------------------------------------------------
# 5.1 Autoencoder model, training helper, and evaluation helper
#------------------------------------------------------------------------------
class SmallAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
        )
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.dec(self.enc(x))


# Train the autoencoder on normal training images only.
def train_ae(model, train_loader, epochs, lr):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = []
    fit_start = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        batch_losses = []

        for images, _, _, _ in train_loader:
            images = images.to(DEVICE, non_blocking=(DEVICE == "cuda"))

            outputs = model(images)
            loss = F.mse_loss(outputs, images)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batch_losses.append(loss.item())

        epoch_loss = float(np.mean(batch_losses)) if batch_losses else np.nan
        history.append(epoch_loss)
        print(f"AE epoch {epoch:02d} | mean_train_loss={epoch_loss:.6f}")

    return {
        "epoch_losses": history,
        "fit_sec": float(time.time() - fit_start),
    }


# Score a batch using reconstruction error and return image scores plus heatmaps.
@torch.inference_mode()
def ae_scores(model, images):
    model.eval()
    images = images.to(DEVICE, non_blocking=(DEVICE == "cuda"))

    recon = model(images)
    err = (recon - images).pow(2).mean(dim=1)

    heatmaps = err.detach().cpu().numpy()
    flat = heatmaps.reshape(heatmaps.shape[0], -1)
    topk = min(PATCH_SCORE_TOPK, flat.shape[1])
    scores = np.mean(np.sort(flat, axis=1)[:, -topk:], axis=1)

    return scores, heatmaps


# Evaluate the autoencoder on the test loader and return the main metrics.
def eval_method(test_loader, score_fn):
    y_true = []
    y_score = []
    masks = []
    heatmaps = []

    eval_start = time.time()
    n_images = 0

    for images, labels, batch_masks, _ in test_loader:
        scores, batch_heatmaps = score_fn(images)

        y_true.extend(labels.numpy().tolist())
        y_score.extend(scores.tolist())

        batch_masks_np = batch_masks.squeeze(1).numpy()
        for idx in range(batch_masks_np.shape[0]):
            masks.append(batch_masks_np[idx])
            heatmaps.append(batch_heatmaps[idx])

        n_images += images.shape[0]

    eval_sec = time.time() - eval_start

    return {
        "img_roc_auc": safe_roc_auc(y_true, y_score),
        "img_pr_auc": safe_pr_auc(y_true, y_score),
        "px_roc_auc": pixel_roc_auc(masks, heatmaps),
        "sec_per_img": eval_sec / max(n_images, 1),
        "n_eval_images": int(n_images),
    }


# 6.0 Run the autoencoder baseline

## Purpose
- train and evaluate the reconstruction baseline for each active category
- save checkpoints, category-level metrics, and mean summaries
- merge the autoencoder rows into a combined baseline table for later comparison notebooks


In [ ]:
#------------------------------------------------------------------------------
# 6.1 Train and evaluate the autoencoder baseline
#------------------------------------------------------------------------------
autoencoder_settings = {
    "run_mode": RUN_MODE,
    "seed": SEED,
    "device": DEVICE,
    "mvtec_dir": str(MVTEC_DIR),
    "split_manifest_path": str(SPLIT_MANIFEST_PATH),
    "imagenet_category_table_path": str(TABLE_BASELINES_IMAGENET_CATEGORY_PATH),
    "categories_active": CATEGORIES_ACTIVE,
    "image_size": IMG_SIZE,
    "batch_size_train": BATCH_SIZE_TRAIN,
    "batch_size_test": BATCH_SIZE_TEST,
    "ae_epochs": AE_EPOCHS,
    "ae_lr": AE_LR,
    "score_topk_pixels": PATCH_SCORE_TOPK,
    "stage": "autoencoder_baseline_only",
}
save_json_overwrite(autoencoder_settings, AUTOENCODER_SETTINGS_PATH)

ordered_cols = [
    "category",
    "method",
    "run_mode",
    "representation_source",
    "anomaly_head",
    "aug_strength",
    "backbone_name",
    "layers",
    "img_size",
    "coreset_ratio",
    "n_train_good",
    "n_val_good",
    "n_test_total",
    "n_test_anomaly",
    "img_roc_auc",
    "img_pr_auc",
    "px_roc_auc",
    "fit_sec",
    "sec_per_img",
    "n_eval_images",
    "feature_dim",
    "memory_bank_n",
    "memory_bank_mb",
    "checkpoint_size_mb",
    "peak_gpu_mem_mb",
]

autoencoder_rows = []
training_rows = []
qual_cache = None

for category in CATEGORIES_ACTIVE:
    print("\n" + "-" * 90)
    print("CATEGORY:", category)
    print("-" * 90)

    train_items = splits[category]["train_good"]
    val_items = splits[category]["val_good"]
    test_items = splits[category]["test"]

    n_test_total = len(test_items)
    n_test_anomaly = int(sum(int(item["label"]) == 1 for item in test_items))

    train_loader, _, test_loader = make_split_loaders(category)

    reset_peak_gpu_memory()
    model = SmallAE()

    history = train_ae(
        model=model,
        train_loader=train_loader,
        epochs=AE_EPOCHS,
        lr=AE_LR,
    )

    checkpoint_path = CHECKPOINTS_DIR / f"ae_{category}.pt"
    torch.save(model.state_dict(), checkpoint_path)

    eval_result = eval_method(test_loader, lambda images: ae_scores(model, images))
    peak_gpu_mem_mb = get_peak_gpu_memory_mb()

    autoencoder_rows.append(
        {
            "category": category,
            "method": "autoencoder",
            "run_mode": RUN_MODE,
            "representation_source": "none",
            "anomaly_head": "autoencoder",
            "aug_strength": "none",
            "backbone_name": "small_conv_ae",
            "layers": "reconstruction_error",
            "img_size": IMG_SIZE,
            "coreset_ratio": np.nan,
            "n_train_good": len(train_items),
            "n_val_good": len(val_items),
            "n_test_total": n_test_total,
            "n_test_anomaly": n_test_anomaly,
            "img_roc_auc": eval_result["img_roc_auc"],
            "img_pr_auc": eval_result["img_pr_auc"],
            "px_roc_auc": eval_result["px_roc_auc"],
            "fit_sec": history["fit_sec"],
            "sec_per_img": eval_result["sec_per_img"],
            "n_eval_images": eval_result["n_eval_images"],
            "feature_dim": np.nan,
            "memory_bank_n": np.nan,
            "memory_bank_mb": np.nan,
            "checkpoint_size_mb": file_size_mb(checkpoint_path),
            "peak_gpu_mem_mb": peak_gpu_mem_mb,
        }
    )

    for epoch_idx, epoch_loss in enumerate(history["epoch_losses"], start=1):
        training_rows.append(
            {
                "category": category,
                "epoch": epoch_idx,
                "train_loss": epoch_loss,
            }
        )

    print(
        "Autoencoder:",
        {
            "img_roc_auc": eval_result["img_roc_auc"],
            "img_pr_auc": eval_result["img_pr_auc"],
            "px_roc_auc": eval_result["px_roc_auc"],
            "fit_sec": history["fit_sec"],
            "sec_per_img": eval_result["sec_per_img"],
        },
    )

    if category == QUAL_CATEGORY:
        qual_items = select_qual_items(test_items, n_good=3, n_bad=3)

        if len(qual_items) > 0:
            qual_loader = make_loader(
                items=qual_items,
                mode="test",
                img_tfm=tfm_ae,
                batch_size=len(qual_items),
                shuffle=False,
            )

            qual_images, qual_labels, qual_masks, qual_paths = next(iter(qual_loader))
            _, qual_heatmaps = ae_scores(model, qual_images)

            qual_cache = {
                "category": category,
                "images": qual_images.cpu(),
                "labels": qual_labels.numpy(),
                "masks": qual_masks.squeeze(1).numpy(),
                "paths": list(qual_paths),
                "heatmaps": qual_heatmaps,
            }

    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

df_baselines_autoencoder_category = pd.DataFrame(autoencoder_rows)[ordered_cols]
df_training_loss = pd.DataFrame(training_rows)

mean_metric_cols = [
    "img_roc_auc",
    "img_pr_auc",
    "px_roc_auc",
    "fit_sec",
    "sec_per_img",
    "feature_dim",
    "memory_bank_n",
    "memory_bank_mb",
    "checkpoint_size_mb",
    "peak_gpu_mem_mb",
]
df_baselines_autoencoder_mean = (
    df_baselines_autoencoder_category
    .groupby(["method", "representation_source", "anomaly_head", "aug_strength"], as_index=False)[mean_metric_cols]
    .mean(numeric_only=True)
)

df_baselines_autoencoder_mean["category"] = "MEAN"
df_baselines_autoencoder_mean["run_mode"] = RUN_MODE
df_baselines_autoencoder_mean["backbone_name"] = "small_conv_ae"
df_baselines_autoencoder_mean["layers"] = "reconstruction_error"
df_baselines_autoencoder_mean["img_size"] = IMG_SIZE
df_baselines_autoencoder_mean["coreset_ratio"] = np.nan
df_baselines_autoencoder_mean["n_train_good"] = np.nan
df_baselines_autoencoder_mean["n_val_good"] = np.nan
df_baselines_autoencoder_mean["n_test_total"] = np.nan
df_baselines_autoencoder_mean["n_test_anomaly"] = np.nan
df_baselines_autoencoder_mean["n_eval_images"] = np.nan

df_baselines_autoencoder_mean = df_baselines_autoencoder_mean[ordered_cols]
df_baselines_autoencoder_full = pd.concat(
    [df_baselines_autoencoder_category, df_baselines_autoencoder_mean],
    ignore_index=True,
)

save_csv_overwrite(df_baselines_autoencoder_category, TABLE_BASELINES_AUTOENCODER_CATEGORY_PATH)
save_csv_overwrite(df_baselines_autoencoder_mean, TABLE_BASELINES_AUTOENCODER_MEAN_PATH)
save_csv_overwrite(df_baselines_autoencoder_full, TABLE_BASELINES_AUTOENCODER_FULL_PATH)

df_baselines_all_category = pd.concat(
    [df_baselines_imagenet_category[ordered_cols], df_baselines_autoencoder_category],
    ignore_index=True,
)

df_baselines_all_mean = (
    df_baselines_all_category
    .groupby(["method", "representation_source", "anomaly_head", "aug_strength"], as_index=False)[mean_metric_cols]
    .mean(numeric_only=True)
)
df_baselines_all_mean["category"] = "MEAN"
df_baselines_all_mean["run_mode"] = RUN_MODE
df_baselines_all_mean["backbone_name"] = ""
df_baselines_all_mean["layers"] = ""
df_baselines_all_mean["img_size"] = IMG_SIZE
df_baselines_all_mean["coreset_ratio"] = np.nan
df_baselines_all_mean["n_train_good"] = np.nan
df_baselines_all_mean["n_val_good"] = np.nan
df_baselines_all_mean["n_test_total"] = np.nan
df_baselines_all_mean["n_test_anomaly"] = np.nan
df_baselines_all_mean["n_eval_images"] = np.nan
df_baselines_all_mean = df_baselines_all_mean[ordered_cols]

df_baselines_all_full = pd.concat(
    [df_baselines_all_category, df_baselines_all_mean],
    ignore_index=True,
)

save_csv_overwrite(df_baselines_all_category, TABLE_BASELINES_ALL_CATEGORY_PATH)
save_csv_overwrite(df_baselines_all_mean, TABLE_BASELINES_ALL_MEAN_PATH)
save_csv_overwrite(df_baselines_all_full, TABLE_BASELINES_ALL_FULL_PATH)

print_banner("Autoencoder tables")
display(df_baselines_autoencoder_full.tail(10))

print_banner("Merged baseline table")
display(df_baselines_all_full.tail(12))


# 7.0 Save baseline figures

## Purpose
- save one compact summary figure for the autoencoder-only result
- save one merged three-method baseline summary figure
- save one qualitative heatmap figure for the selected category
- save one training-loss figure so the reconstruction stage is easier to inspect


In [ ]:
#------------------------------------------------------------------------------
# 7.1 Save summary, training, and qualitative figures
#------------------------------------------------------------------------------
method_order_all = ["imagenet_patchcore", "imagenet_padim", "autoencoder"]

# Autoencoder-only summary figure.
mean_plot_df_ae = (
    df_baselines_autoencoder_category
    .groupby("method")[["img_roc_auc", "img_pr_auc", "px_roc_auc", "sec_per_img"]]
    .mean(numeric_only=True)
    .reset_index()
)

fig = plt.figure(figsize=(12, 8))

ax1 = plt.subplot(2, 2, 1)
ax1.bar(mean_plot_df_ae["method"], mean_plot_df_ae["img_roc_auc"])
ax1.set_title("Mean image ROC-AUC")
ax1.set_ylim(0, 1.02)
ax1.tick_params(axis="x", rotation=20)

ax2 = plt.subplot(2, 2, 2)
ax2.bar(mean_plot_df_ae["method"], mean_plot_df_ae["img_pr_auc"])
ax2.set_title("Mean image PR-AUC")
ax2.set_ylim(0, 1.02)
ax2.tick_params(axis="x", rotation=20)

ax3 = plt.subplot(2, 2, 3)
ax3.bar(mean_plot_df_ae["method"], mean_plot_df_ae["px_roc_auc"])
ax3.set_title("Mean pixel ROC-AUC")
ax3.set_ylim(0, 1.02)
ax3.tick_params(axis="x", rotation=20)

ax4 = plt.subplot(2, 2, 4)
ax4.bar(mean_plot_df_ae["method"], mean_plot_df_ae["sec_per_img"])
ax4.set_title("Mean inference sec / image")
ax4.tick_params(axis="x", rotation=20)

plt.tight_layout()
ensure_parent(FIG_BASELINES_AUTOENCODER_PATH)
plt.savefig(FIG_BASELINES_AUTOENCODER_PATH, dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Saved figure:", FIG_BASELINES_AUTOENCODER_PATH)

# Merged three-method baseline summary figure.
mean_plot_df_all = (
    df_baselines_all_category
    .groupby("method")[["img_roc_auc", "img_pr_auc", "px_roc_auc", "sec_per_img"]]
    .mean(numeric_only=True)
    .reindex(method_order_all)
    .reset_index()
)

fig = plt.figure(figsize=(12, 8))

ax1 = plt.subplot(2, 2, 1)
ax1.bar(mean_plot_df_all["method"], mean_plot_df_all["img_roc_auc"])
ax1.set_title("Mean image ROC-AUC")
ax1.set_ylim(0, 1.02)
ax1.tick_params(axis="x", rotation=20)

ax2 = plt.subplot(2, 2, 2)
ax2.bar(mean_plot_df_all["method"], mean_plot_df_all["img_pr_auc"])
ax2.set_title("Mean image PR-AUC")
ax2.set_ylim(0, 1.02)
ax2.tick_params(axis="x", rotation=20)

ax3 = plt.subplot(2, 2, 3)
ax3.bar(mean_plot_df_all["method"], mean_plot_df_all["px_roc_auc"])
ax3.set_title("Mean pixel ROC-AUC")
ax3.set_ylim(0, 1.02)
ax3.tick_params(axis="x", rotation=20)

ax4 = plt.subplot(2, 2, 4)
ax4.bar(mean_plot_df_all["method"], mean_plot_df_all["sec_per_img"])
ax4.set_title("Mean inference sec / image")
ax4.tick_params(axis="x", rotation=20)

plt.tight_layout()
ensure_parent(FIG_BASELINES_ALL_PATH)
plt.savefig(FIG_BASELINES_ALL_PATH, dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Saved figure:", FIG_BASELINES_ALL_PATH)

# Training-loss figure.
if len(df_training_loss) > 0:
    loss_plot_df = (
        df_training_loss
        .groupby("epoch", as_index=False)["train_loss"]
        .mean()
        .sort_values("epoch")
        .reset_index(drop=True)
    )

    fig = plt.figure(figsize=(7, 4))
    plt.plot(loss_plot_df["epoch"], loss_plot_df["train_loss"], marker="o")
    plt.title("Mean autoencoder training loss")
    plt.xlabel("Epoch")
    plt.ylabel("Train loss")
    plt.tight_layout()
    ensure_parent(FIG_AE_TRAINING_LOSS_PATH)
    plt.savefig(FIG_AE_TRAINING_LOSS_PATH, dpi=220, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print("Saved figure:", FIG_AE_TRAINING_LOSS_PATH)
else:
    print("Training-loss figure was skipped because no training rows were saved.")

# Qualitative heatmap figure.
if qual_cache is not None:
    qual_images = qual_cache["images"]
    qual_labels = qual_cache["labels"]
    qual_masks = qual_cache["masks"]
    qual_paths = qual_cache["paths"]
    qual_heatmaps = qual_cache["heatmaps"]

    row_n = qual_images.shape[0]
    plt.figure(figsize=(9, 3.0 * row_n))

    for row_idx in range(row_n):
        row_name = f"{'anomaly' if qual_labels[row_idx] == 1 else 'good'} | {Path(qual_paths[row_idx]).parent.name}"

        ax = plt.subplot(row_n, 3, row_idx * 3 + 1)
        overlay(ax, qual_images[row_idx], None, title="Image")
        ax.set_ylabel(row_name, fontsize=9)

        ax = plt.subplot(row_n, 3, row_idx * 3 + 2)
        show_mask(ax, qual_masks[row_idx], title="GT mask")

        ax = plt.subplot(row_n, 3, row_idx * 3 + 3)
        overlay(ax, qual_images[row_idx], qual_heatmaps[row_idx], title="AE heatmap")

    plt.tight_layout()
    ensure_parent(FIG_HEATMAPS_AUTOENCODER_PATH)
    plt.savefig(FIG_HEATMAPS_AUTOENCODER_PATH, dpi=220, bbox_inches="tight")
    plt.show()
    plt.close()

    print("Saved figure:", FIG_HEATMAPS_AUTOENCODER_PATH)
else:
    print("No qualitative cache was created, so the heatmap figure was skipped.")


# 8.0 Final review

## Purpose
- list the saved artefacts clearly
- confirm which prior notebooks this notebook depends on
- make GitHub upload and later notebook reuse simpler


In [ ]:
#------------------------------------------------------------------------------
# 8.1 Final saved artefacts
#------------------------------------------------------------------------------
print_banner("Saved artefacts")
print("Run metadata path             :", RUN_METADATA_PATH)
print("Autoencoder settings path     :", AUTOENCODER_SETTINGS_PATH)
print("AE category table path        :", TABLE_BASELINES_AUTOENCODER_CATEGORY_PATH)
print("AE mean table path            :", TABLE_BASELINES_AUTOENCODER_MEAN_PATH)
print("AE full table path            :", TABLE_BASELINES_AUTOENCODER_FULL_PATH)
print("Merged category table path    :", TABLE_BASELINES_ALL_CATEGORY_PATH)
print("Merged mean table path        :", TABLE_BASELINES_ALL_MEAN_PATH)
print("Merged full table path        :", TABLE_BASELINES_ALL_FULL_PATH)
print("AE summary figure path        :", FIG_BASELINES_AUTOENCODER_PATH)
print("Merged summary figure path    :", FIG_BASELINES_ALL_PATH)
print("AE training-loss figure path  :", FIG_AE_TRAINING_LOSS_PATH)
print("AE qualitative figure path    :", FIG_HEATMAPS_AUTOENCODER_PATH)
print("Checkpoint folder             :", CHECKPOINTS_DIR)
print("Split manifest dependency     :", SPLIT_MANIFEST_PATH)
print("Leakage report dependency     :", LEAKAGE_REPORT_PATH)
print("ImageNet table dependency     :", TABLE_BASELINES_IMAGENET_CATEGORY_PATH)
